# Normalisasi Nama Produk Berbasis Master Referensi

Notebook ini digunakan untuk menormalisasi nama produk transaksi berdasarkan file master `master_produk_brand.csv`. Metode dilakukan bertahap: **pembersihan teks**, **pencocokan persis**, **pencocokan berbasis token**, lalu **fuzzy matching** menggunakan `rapidfuzz`.

catatan: sistem tidak boleh memaksakan hasil ketika tingkat kemiripan rendah. Baris dengan status `REVIEW` harus divalidasi terhadap patokan/label pembanding sebelum digunakan dalam analisis skripsi.

## 1. Instalasi dan impor pustaka

`pandas` digunakan untuk pengolahan tabel, `re` dan `unicodedata` untuk standardisasi teks, sedangkan `rapidfuzz` digunakan untuk menghitung kemiripan nama produk secara efisien.

In [1]:
# Jalankan satu kali jika rapidfuzz belum tersedia
# %pip install pandas rapidfuzz openpyxl

import re
import unicodedata
from pathlib import Path

import pandas as pd
from rapidfuzz import process, fuzz

pd.set_option("display.max_colwidth", 120)


## 2. Konfigurasi lokasi file

Ubah `FILE_TRANSAKSI` apabila file yang akan dinormalisasi memiliki nama atau lokasi berbeda. File master yang diberikan berisi kolom `product_id`, `brand`, dan `produk_standar`. File transaksi minimal harus memiliki satu kolom nama produk.

In [2]:

FILE_MASTER = Path("master_produk_brand.xlsx")

# Ganti dengan file transaksi 
FILE_TRANSAKSI = Path("data_transaksi.csv")  # Isi dengan Path("nama_file.csv") atau Path("nama_file.xlsx")

# Nama kolom produk pada file transaksi. Jika None, kode akan mencoba mendeteksinya.
KOLOM_PRODUK = "produk_bersih"

# Ambang keputusan;
AMBANG_TERIMA = 88
AMBANG_REVIEW = 75

# Alias manual dapat ditambah setelah memperoleh patokan pembanding.
# Format: nama setelah pembersihan -> produk_standar master
ALIAS_MANUAL = {
     "azarine oren": "AZARINE ORANGE",
     "AZARINE OREN": "AZARINE ORANGE",
     "AZARINE BIRU IJO": "AZARINE BIRU KECIL",
     "AZARINE 50 ML": "AZARINE 50 ML",
     "AZARINE 50 ML IJO": "AZARINE 50 ML",
     "AZARINE 50 ML SPF 45": "AZARINE 50 ML",
     "AZARINE SPF 45 HYDRA": "AZARINE HYDRASHOOT",

     "PURBASARI WHITEGLOW": "PURBASARI WHITEGLOW",
     "PURBASARI WHITGLOW": "PURBASARI WHITEGLOW",
     "PURBASARI. CYBRGT": "PURBASARI BODYSERUM",
     "PRECIOUS SKIN AHA 250 ML": "AHA PRECIOUS LOTION",
     "AHA PRECIOUS 100 ML": "AHA PRECIOUS LOTION",
     "P.LULUR 500 GR": "PURBASARI LULUR 500 GR",
     "MOIST BRIGHT ORIGINOTE": "MOIST BRIGHT ORIGINOTE",
     "SOAP NALPAMARA": "BODYWASH NALPAMARA",
     "VASELINE 100 ML": "VASELINE BODYSERUM",
     "VASELIENE 200 ML": "VASELINE BODYSERUM",
     "VASELINE 200 ML" : "VASELINE BODYSERUM",
     "VASELINE 330 ML": "VASELINE BODYSERUM",
     "VASELINE GOLD 330 ML": "VASELINE BODYSERUM",
     "VASELINE PINK 330 ML": "VASELINE BODYSERUM",
     "VASELINE BIRU 330 ML": "VASELINE BODYSERUM",
     "VASELLINE 330 ML": "VASELINE BODYSERUM",
     "VASELINE BODYSERUM":"VASELINE 100 ML EXP 2027"

     

     

}


## 3. Fungsi pembersihan dan standardisasi teks

Fungsi berikut melakukan: konversi huruf besar, penghapusan aksen, penggantian simbol menjadi spasi, penggabungan spasi berlebih, serta normalisasi beberapa satuan umum. Tahap ini tidak mengubah label standar pada master; tahap ini hanya membuat pembandingan lebih konsisten.

In [3]:
def bersihkan_teks(nilai):
    """Menghasilkan bentuk teks yang konsisten untuk proses pencocokan."""
    if pd.isna(nilai):
        return ""
    teks = str(nilai).strip().upper()
    teks = unicodedata.normalize("NFKD", teks)
    teks = "".join(k for k in teks if not unicodedata.combining(k))
    teks = teks.replace("&", " DAN ")
    #menghapus karakter khusus, kecuali spasi dan huruf/angka
    teks = re.sub(r"[\/|_,;:()\[\]{}+\-]", " ", teks)
    teks = re.sub(r"\bMILLI?L?ITER\b|\bML\b", " ML ", teks)
    teks = re.sub(r"\bGRAM\b|\bGR\b", " GR ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    #mengganti beberapa kata yang umum digunakan dalam produk kosmetik
    return teks

def token_sort(teks):
    """Mengurutkan token agar perbedaan posisi kata tidak terlalu berpengaruh."""
    return " ".join(sorted(teks.split()))


## 4. Membaca dan memvalidasi master

Validasi ini penting untuk memastikan tidak ada kolom wajib yang hilang. Jika terdapat `produk_standar` ganda dengan isi berbeda, data master harus diperbaiki terlebih dahulu.

**Catatan hasil pemeriksaan:** Excel patokan berisi 510 baris dan 198 label `produk_standar` unik secara mentah. Setelah pembersihan spasi/format, beberapa label identik diperlakukan sebagai satu label standar. Pengulangan label pada beberapa `product_id` dipertahankan karena merupakan bagian dari pemetaan patokan.


In [6]:
master = (pd.read_excel(FILE_MASTER, dtype={"product_id": "Int64"})
          if FILE_MASTER.suffix.lower() in [".xlsx", ".xls"]
          else pd.read_csv(FILE_MASTER, encoding="utf-8-sig", dtype={"product_id": "Int64"}))
wajib_master = {"product_id", "brand", "produk_standar"}
hilang = wajib_master - set(master.columns)
if hilang:
    raise ValueError(f"Kolom master yang hilang: {sorted(hilang)}")

master = master.dropna(subset=["produk_standar"]).copy()
master["produk_standar"] = master["produk_standar"].astype(str).str.strip()
master["kunci_produk"] = master["produk_standar"].map(bersihkan_teks)
master["kunci_token"] = master["kunci_produk"].map(token_sort)

# Satu kunci harus menunjuk ke satu label standar agar keputusan tidak ambigu.
cek_ganda = (master.groupby("kunci_produk")["produk_standar"].nunique() > 1).sum()
print(f"Jumlah baris master      : {len(master):,}")
print(f"Jumlah produk standar    : {master['produk_standar'].nunique():,}")
print(f"Jumlah merek             : {master['brand'].nunique():,}")
print(f"Kunci dengan konflik     : {cek_ganda:,}")


Jumlah baris master      : 510
Jumlah produk standar    : 181
Jumlah merek             : 46
Kunci dengan konflik     : 3


## 5. Pemetaan produk

Urutan keputusan: (1) alias manual, (2) kecocokan persis, (3) kecocokan token, (4) fuzzy matching.

- `MATCH_EXACT`: kecocokan persis setelah pembersihan.
- `MATCH_TOKEN`: kata-kata sama tetapi urutannya berbeda.
- `MATCH_FUZZY`: kecocokan berdasarkan skor kemiripan.
- `REVIEW`: skor belum cukup tinggi atau kandidat pertama dan kedua terlalu dekat.
- `UNMATCHED`: nama kosong atau tidak memiliki kandidat yang layak.

In [8]:
# Kamus yang dipakai pada proses pencocokan
standar_unik = master.drop_duplicates("kunci_produk").copy()
map_exact = dict(zip(standar_unik["kunci_produk"], standar_unik["produk_standar"]))
map_token = dict(zip(standar_unik["kunci_token"], standar_unik["produk_standar"]))
choices = standar_unik["kunci_produk"].tolist()

def normalisasi_satu(nilai):
    asli = "" if pd.isna(nilai) else str(nilai)
    kunci = bersihkan_teks(asli)
    hasil = {
        "produk_asli": asli,
        "produk_bersih": kunci,
        "produk_standar": pd.NA,
        "skor_kemiripan": pd.NA,
        "skor_kedua": pd.NA,
        "metode": "UNMATCHED",
        "status": "UNMATCHED",
    }
    if not kunci:
        return hasil

    if kunci in ALIAS_MANUAL:
        hasil.update(produk_standar=ALIAS_MANUAL[kunci], skor_kemiripan=100, metode="ALIAS_MANUAL", status="MATCH")
        return hasil

    if kunci in map_exact:
        hasil.update(produk_standar=map_exact[kunci], skor_kemiripan=100, skor_kedua=0, metode="MATCH_EXACT", status="MATCH")
        return hasil

    tk = token_sort(kunci)
    if tk in map_token:
        hasil.update(produk_standar=map_token[tk], skor_kemiripan=99, skor_kedua=0, metode="MATCH_TOKEN", status="MATCH")
        return hasil

    kandidat = process.extract(kunci, choices, scorer=fuzz.WRatio, limit=2)
    if not kandidat:
        return hasil
    kunci_1, skor_1, _ = kandidat[0]
    skor_2 = kandidat[1][1] if len(kandidat) > 1 else 0
    hasil.update(
        produk_standar=map_exact[kunci_1],
        skor_kemiripan=round(skor_1, 2),
        skor_kedua=round(skor_2, 2),
        metode="MATCH_FUZZY",
        status=("MATCH" if skor_1 >= AMBANG_TERIMA and (skor_1 - skor_2) >= 3 else
                "REVIEW" if skor_1 >= AMBANG_REVIEW else "UNMATCHED")
    )
    if hasil["status"] == "UNMATCHED":
        hasil["produk_standar"] = pd.NA
    return hasil

def normalisasi_dataframe(df, kolom_produk):
    hasil = pd.DataFrame([normalisasi_satu(x) for x in df[kolom_produk]])
    return pd.concat([df.reset_index(drop=True), hasil.drop(columns=["produk_asli"])], axis=1)


## 6. Menjalankan normalisasi pada data transaksi

Kode ini mencoba mengenali nama kolom produk secara otomatis. Jika gagal, tetapkan `KOLOM_PRODUK` secara eksplisit. Kolom transaksi asli dipertahankan agar hasil dapat diaudit.

In [10]:
def deteksi_kolom_produk(df):#fungsi untuk mendeteksi kolom produk pada dataframe
    kandidat = ["produk", "nama_produk", "product", "product_name", "item", "nama_barang","produk_bersih"]
    ditemukan = [c for c in kandidat if c in df.columns]
    if ditemukan:
        return ditemukan[0]
    raise ValueError(f"Kolom produk tidak terdeteksi. Kolom tersedia: {df.columns.tolist()}")

if FILE_TRANSAKSI is not None and FILE_TRANSAKSI.exists():#jika file transaksi tersedia, maka akan dibaca dan diproses
    transaksi = (pd.read_excel(FILE_TRANSAKSI) if FILE_TRANSAKSI.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(FILE_TRANSAKSI, encoding="utf-8-sig"))
    kolom = KOLOM_PRODUK or deteksi_kolom_produk(transaksi)
    hasil_normalisasi = normalisasi_dataframe(transaksi, kolom)

    ringkasan = hasil_normalisasi["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="jumlah")
    display(ringkasan)
    display(hasil_normalisasi.head(10))
else:
    print("FILE_TRANSAKSI belum diisi. Tetapkan lokasi file transaksi terlebih dahulu.")
    print("Isi FILE_TRANSAKSI setelah Anda menyiapkan data transaksi.")


,status,jumlah
0,MATCH,2374
1,UNMATCHED,61
2,REVIEW,58


,bulan,tanggal,no_transaksi,pelanggan,qty,harga_beli,total_beli,harga_jual,total_jual,keuntungan,brand,produk_asli,produk_bersih,produk_bersih,produk_standar,skor_kemiripan,skor_kedua,metode,status
0,JANUARI,2026-01-01,1,LUTFINUR,72,14000.0,1008000.0,14500,1044000,36000.0,PURBASARI,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100,0,MATCH_EXACT,MATCH
1,JANUARI,2026-01-02,1,HARIS,150,35000.0,5250000.0,36500,5475000,225000.0,SKINAQUA,SKINAQUA PUTIH,SKINAQUA PUTIH,SKINAQUA PUTIH,NaN,69.09,69.09,MATCH_FUZZY,UNMATCHED
2,JANUARI,2026-01-02,1,HARIS,144,16500.0,2376000.0,18000,2592000,216000.0,PERFECT,PERFECT WHITE AHA 50 ML,PERFECT WHITE AHA 50 ML,PERFECT WHITE AHA 50 ML,PERFECT WHITE AHA 50 ML,100,0,MATCH_EXACT,MATCH
3,JANUARI,2026-01-02,2,KEDAIMUE,192,14000.0,2688000.0,14500,2784000,96000.0,PURBASARI,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100,0,MATCH_EXACT,MATCH
4,JANUARI,2026-01-02,2,KEDAIMUE,96,29000.0,2784000.0,34500,3312000,528000.0,G2G,MICELLAR G2G 300 ML,G2G MICELLAR 300 ML PINK,G2G MICELLAR 300 ML PINK,G2G MICELLAR 300 ML PINK,100,0,MATCH_EXACT,MATCH
5,JANUARI,2026-01-02,3,SITA,360,14000.0,5040000.0,14500,5220000,180000.0,PURBASARI,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100,0,MATCH_EXACT,MATCH
6,JANUARI,2026-01-02,4,WAFA,180,19000.0,3420000.0,20000,3600000,180000.0,PURBASARI,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,100,0,MATCH_EXACT,MATCH
7,JANUARI,2026-01-02,5,TERE,2376,15300.0,36352800.0,15700,37303200,950400.0,HANASUI,LIPCREAM HANASUI,HANASUI LIPCREAM MATCHA,HANASUI LIPCREAM MATCHA,HANASUI LIPCREAM MATCHA,100,0,MATCH_EXACT,MATCH
8,JANUARI,2026-01-02,6,SAZIRAH,192,24000.0,4608000.0,26000,4992000,384000.0,G2G,PEELING G2G,AMINO CLEANSER G2G,AMINO CLEANSER G2G,AMINO CLEANSER G2G,100,0,MATCH_EXACT,MATCH
9,JANUARI,2026-01-02,7,FENITA,60,19000.0,1140000.0,20000,1200000,60000.0,PURBASARI,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,100,0,MATCH_EXACT,MATCH


## 7. Meninjau hasil yang belum pasti

Baris `REVIEW` harus dibandingkan dengan patokan manual. Jangan memasukkannya ke analisis final tanpa keputusan yang terdokumentasi.

In [13]:
if "hasil_normalisasi" in globals():
    perlu_review = hasil_normalisasi[hasil_normalisasi["status"].isin(["REVIEW", "UNMATCHED"])].copy()
    display(perlu_review.sort_values(["status", "skor_kemiripan"], ascending=[True, False]))

    hasil_normalisasi.to_csv("hasil_normalisasi_sementara.csv", index=False, encoding="utf-8-sig")
    print("File tersimpan: hasil_normalisasi_sementara.csv")


,bulan,tanggal,no_transaksi,pelanggan,qty,harga_beli,total_beli,harga_jual,total_jual,keuntungan,brand,produk_asli,produk_bersih,produk_bersih,produk_standar,skor_kemiripan,skor_kedua,metode,status
49,JANUARI,2026-01-05,16,HARIS,96,25000.0,2400000.0,26000,2496000,96000.0,G2G,MOIST BLUE G2G,MOIST BLUE G2G,MOIST BLUE G2G,FFY MOIST,85.5,85.5,MATCH_FUZZY,REVIEW
58,JANUARI,2026-01-06,10,DIAZ,96,25000.0,2400000.0,26000,2496000,96000.0,G2G,MOIST BLUE G2G,MOIST BLUE G2G,MOIST BLUE G2G,FFY MOIST,85.5,85.5,MATCH_FUZZY,REVIEW
277,JANUARI,2026-01-20,7,SANTI,96,25000.0,2400000.0,26000,2496000,96000.0,G2G,MOIST BLUE G2G,MOIST BLUE G2G,MOIST BLUE G2G,FFY MOIST,85.5,85.5,MATCH_FUZZY,REVIEW
306,JANUARI,2026-01-22,14,DIAZ,96,25000.0,2400000.0,26000,2496000,96000.0,G2G,MOIST BLUE G2G,MOIST BLUE G2G,MOIST BLUE G2G,FFY MOIST,85.5,85.5,MATCH_FUZZY,REVIEW
317,JANUARI,2026-01-24,5,GUDANG JEMBER,192,22500.0,4320000.0,25000,4800000,480000.0,G2G,MOIST BLUE G2G,MOIST BLUE G2G,MOIST BLUE G2G,FFY MOIST,85.5,85.5,MATCH_FUZZY,REVIEW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2132,JULI,2026-07-31,1,YULI,400,33000.0,13200000.0,35000,14000000,800000.0,AZARINE,AZARINE SPF 45,NaN,,NaN,<NA>,<NA>,UNMATCHED,UNMATCHED
2147,AGUSTUS,2026-08-05,1,ANTON,100,33000.0,3300000.0,36000,3600000,300000.0,AZARINE,AZARINE SPF 45,NaN,,NaN,<NA>,<NA>,UNMATCHED,UNMATCHED
2185,AGUSTUS,2026-08-12,2,FURQON,100,19000.0,1900000.0,23000,2300000,400000.0,AZARINE,AZARINE IJO SPF 45,NaN,,NaN,<NA>,<NA>,UNMATCHED,UNMATCHED
2314,AGUSTUS,2026-08-27,3,SUKMA,96,32500.0,3120000.0,34000,3264000,144000.0,SKINAQUA,SKINAQUA PUTIH,NaN,,NaN,<NA>,<NA>,UNMATCHED,UNMATCHED


File tersimpan: hasil_normalisasi_sementara.csv


8. membuat list brand dan produk yang unik


In [15]:
import pandas as pd

# 1. Membaca file hasil normalisasi
file_input = "hasil_normalisasi_sementara.csv"

df = pd.read_csv(file_input, encoding="utf-8-sig")

# 2. Sesuaikan nama kolom
kolom_brand = "brand"
kolom_produk = "produk_bersih"

# 3. Membersihkan teks
df[kolom_brand] = (
    df[kolom_brand]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df[kolom_produk] = (
    df[kolom_produk]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# 4. Membuat list brand unik
list_brand = (
    df[[kolom_brand]]
    .dropna()
    .drop_duplicates()
    .rename(columns={kolom_brand: "brand"})
    .sort_values("brand")
    .reset_index(drop=True)
)

list_brand.insert(0, "no", range(1, len(list_brand) + 1))

# 5. Membuat list produk unik
list_produk = (
    df[[kolom_produk]]
    .dropna()
    .drop_duplicates()
    .rename(columns={kolom_produk: "produk"})
    .sort_values("produk")
    .reset_index(drop=True)
)

list_produk.insert(0, "no", range(1, len(list_produk) + 1))

# 6. Menampilkan hasil
print("DAFTAR BRAND UNIK")
display(list_brand)

print("DAFTAR PRODUK UNIK")
display(list_produk)

# 7. Menyimpan hasil ke file terpisah
list_brand.to_csv(
    "list_brand_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

list_produk.to_csv(
    "list_produk_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

# Alternatif penyimpanan ke Excel
list_brand.to_excel(
    "list_brand_unique.xlsx",
    index=False
)

list_produk.to_excel(
    "list_produk_unique.xlsx",
    index=False
)

print(f"Jumlah brand unik  : {len(list_brand)}")
print(f"Jumlah produk unik : {len(list_produk)}")


DAFTAR BRAND UNIK


,no,brand
0,1,AZARINE
1,2,BEAUTYNU
2,3,BRYLE
3,4,BRYLEE
4,5,CINDYNAL
5,6,DAZZLE ME
6,7,EMINA
7,8,ESKULIN
8,9,FACETOLOGY
9,10,FFY


DAFTAR PRODUK UNIK


,no,produk
0,1,AHA PRECIOUS LOTION
1,2,AMINO CLEANSER G2G
2,3,AZARINE 50 ML
3,4,AZARINE BIRU KECIL
4,5,AZARINE HIJAU
...,...,...
161,162,VASELINE BODYLOTION
162,163,WHITENIC BODYLOTION
163,164,WHITENIC BODYMASK
164,165,WHITENIC BODYSERUM


Jumlah brand unik  : 46
Jumlah produk unik : 166


9. PEMECAHAN FITUR UNTUK KOLOM TANGGAL UNTUK XGBOOST
MENJADI KOLOM
tahun
bulan
hari
hari_dalam_minggu



In [17]:
df = pd.read_csv("hasil_normalisasi_sementara.csv", encoding="utf-8-sig")

df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")

df["tahun"] = df["tanggal"].dt.year
df["bulan"] = df["tanggal"].dt.month
df["hari"] = df["tanggal"].dt.day
df["hari_dalam_minggu"] = df["tanggal"].dt.dayofweek

pd.DataFrame(df.head(10))

df.to_csv("hasil_pecah_tanggal.csv", index=False, encoding="utf-8-sig")
print("File tersimpan: hasil_pecah_tanggal.csv")

File tersimpan: hasil_pecah_tanggal.csv


10. ENCODING UNTUK DATA NON-NUMERICAL UNTUK XGBOOST

In [18]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor


In [19]:
file_input = "hasil_pecah_tanggal.csv"

df = pd.read_csv(
    file_input,
    encoding="utf-8-sig"
)

print("Ukuran data:", df.shape)
print(df.head())
print(df.columns.tolist())


Ukuran data: (2493, 22)
   bulan     tanggal  no_transaksi pelanggan  qty  harga_beli  total_beli  \
0      1  2026-01-01             1  LUTFINUR   72     14000.0   1008000.0   
1      1  2026-01-02             1     HARIS  150     35000.0   5250000.0   
2      1  2026-01-02             1     HARIS  144     16500.0   2376000.0   
3      1  2026-01-02             2  KEDAIMUE  192     14000.0   2688000.0   
4      1  2026-01-02             2  KEDAIMUE   96     29000.0   2784000.0   

   harga_jual  total_jual  keuntungan  ...             produk_bersih  \
0       14500     1044000     36000.0  ...       PURBASARI BODYSERUM   
1       36500     5475000    225000.0  ...            SKINAQUA PUTIH   
2       18000     2592000    216000.0  ...   PERFECT WHITE AHA 50 ML   
3       14500     2784000     96000.0  ...       PURBASARI BODYSERUM   
4       34500     3312000    528000.0  ...  G2G MICELLAR 300 ML PINK   

            produk_bersih.1            produk_standar skor_kemiripan  \
0       

11.TARGET KOLOM

In [ ]:
kolom_tanggal_numerik = [
    "tahun",
    "bulan",
    "hari",
    "hari_dalam_minggu"
]
#fungsi untuk mengubah kolom tanggal menjadi numerik
for kolom in kolom_tanggal_numerik:
    if kolom in df.columns:
        df[kolom] = pd.to_numeric(
            df[kolom],
            errors="coerce"
        )


12. MENENTUKAN TARGET YANG DIPREDIKSI
target yang diprediksi adalah qty

In [21]:
target = "qty"

kolom_numerik = [
    "tahun",
    "bulan",
    "hari",
    "hari_dalam_minggu",
    "harga_beli",
    "harga_jual"
]

kolom_kategorikal = [
    "pelanggan",
    "brand",
    "produk_bersih"
]
#memastikan kolom target ada di data
kolom_diperlukan = (
    [target]
    + kolom_numerik
    + kolom_kategorikal
)

kolom_tidak_ada = [
    kolom for kolom in kolom_diperlukan
    if kolom not in df.columns
]

if kolom_tidak_ada:
    raise ValueError(
        f"Kolom berikut tidak ditemukan: {kolom_tidak_ada}"
    )


13. MEMBERSIHKAN KOLOM TARGET
Nilai qty harus berupa angka. Baris dengan target kosong dikeluarkan karena model tidak dapat belajar dari target yang tidak tersedia.

In [23]:
df[target] = pd.to_numeric(
    df[target],
    errors="coerce"
)

df = df.dropna(
    subset=[target]
).copy()

# Qty tidak boleh negatif
df = df[df[target] >= 0].copy()


13.MEMBAGI DATA MENJADI TRAIN TEST
train 80%
test 20%

In [24]:
batas_train = int(len(df) * 0.8)

data_train = df.iloc[:batas_train].copy()
data_test = df.iloc[batas_train:].copy()

X_train = data_train[
    kolom_numerik + kolom_kategorikal
]

y_train = data_train[target]

X_test = data_test[
    kolom_numerik + kolom_kategorikal
]

y_test = data_test[target]

print("Data training:", X_train.shape)
print("Data testing :", X_test.shape)

pd.DataFrame(X_train.head(10))


Data training: (1994, 9)
Data testing : (499, 9)


,tahun,bulan,hari,hari_dalam_minggu,harga_beli,harga_jual,pelanggan,brand,produk_bersih
0,2026,1,1,3,14000.0,14500,LUTFINUR,PURBASARI,PURBASARI BODYSERUM
1,2026,1,2,4,35000.0,36500,HARIS,SKINAQUA,SKINAQUA PUTIH
2,2026,1,2,4,16500.0,18000,HARIS,PERFECT,PERFECT WHITE AHA 50 ML
3,2026,1,2,4,14000.0,14500,KEDAIMUE,PURBASARI,PURBASARI BODYSERUM
4,2026,1,2,4,29000.0,34500,KEDAIMUE,G2G,G2G MICELLAR 300 ML PINK
5,2026,1,2,4,14000.0,14500,SITA,PURBASARI,PURBASARI BODYSERUM
6,2026,1,2,4,19000.0,20000,WAFA,PURBASARI,PURBASARI LULUR 500 GR
7,2026,1,2,4,15300.0,15700,TERE,HANASUI,HANASUI LIPCREAM MATCHA
8,2026,1,2,4,24000.0,26000,SAZIRAH,G2G,AMINO CLEANSER G2G
9,2026,1,2,4,19000.0,20000,FENITA,PURBASARI,PURBASARI LULUR 500 GR


14. PREPROCESSING DENGAN ColoumnTransformer
Fitur numerik
Nilai kosong pada fitur numerik diisi menggunakan median.
Fitur kategorikal
Nilai kosong pada kolom teks diisi dengan "TIDAK DIKETAHUI", kemudian diubah menggunakan OneHotEncoder.
handle_unknown="ignore" penting agar kategori baru pada data testing tidak menyebabkan error.

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerik",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        )
                    )
                ]
            ),
            kolom_numerik
        ),
        (
            "kategorikal",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="TIDAK DIKETAHUI"
                        )
                    ),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=True
                        )
                    )
                ]
            ),
            kolom_kategorikal
        )
    ],
    remainder="drop"
)


15. MEMBUAT MODEL XGBOOST

Penjelasan parameter utama:
n_estimators: jumlah pohon keputusan.
learning_rate: kecepatan pembelajaran.
max_depth: kedalaman maksimum pohon.
subsample: proporsi data untuk setiap pohon.
colsample_bytree: proporsi fitur untuk setiap pohon.
objective="reg:squarederror": digunakan untuk masalah regresi.

In [30]:
model_xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,# jumlah pohon keputusan
    learning_rate=0.05,# laju pembelajaran
    max_depth=6,# kedalaman maksimum pohon
    subsample=0.8,# proporsi data yang digunakan untuk melatih setiap pohon
    colsample_bytree=0.8,# proporsi fitur yang digunakan untuk melatih setiap pohon
    random_state=42,
    n_jobs=-1
)


16. Menggabungkan preprocessing dan XGBoost

In [31]:
pipeline_xgb = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            model_xgb
        )
    ]
)


17. Melatih model

In [32]:
pipeline_xgb.fit(
    X_train,
    y_train
)

print("Model berhasil dilatih.")


Model berhasil dilatih.


18. MEMBUAT PREDIKSI

In [34]:
y_pred = pipeline_xgb.predict(X_test)

# Quantity tidak seharusnya negatif
y_pred = np.maximum(
    y_pred,
    0
)

hasil_prediksi = data_test.copy()

hasil_prediksi["qty_aktual"] = y_test.values
hasil_prediksi["qty_prediksi"] = y_pred

hasil_prediksi.head()


,bulan,tanggal,no_transaksi,pelanggan,qty,harga_beli,total_beli,harga_jual,total_jual,keuntungan,...,produk_standar,skor_kemiripan,skor_kedua,metode,status,tahun,hari,hari_dalam_minggu,qty_aktual,qty_prediksi
1994,6,2026-06-29,1,ELVIA,64,33000.0,2112000.0,47000,3008000,896000.0,...,SUNSCREEN MOEL,100.0,0.0,MATCH_EXACT,MATCH,2026,29,0,64,105.667099
1995,6,2026-06-29,3,APRIL,115,33000.0,3795000.0,44000,5060000,1265000.0,...,FACE CREAM MOEL,100.0,0.0,MATCH_EXACT,MATCH,2026,29,0,115,105.667099
1996,6,2026-06-29,3,APRIL,51,33000.0,1683000.0,47000,2397000,714000.0,...,SUNSCREEN MOEL,100.0,0.0,MATCH_EXACT,MATCH,2026,29,0,51,105.667099
1997,6,2026-06-30,1,TIA,480,22000.0,10560000.0,25000,12000000,1440000.0,...,G2G MOIST POME,100.0,0.0,MATCH_EXACT,MATCH,2026,30,1,480,218.844543
1998,6,2026-06-30,2,FENITA,96,42000.0,4032000.0,49000,4704000,672000.0,...,MOIST PINK SKINTIFIC,100.0,0.0,MATCH_EXACT,MATCH,2026,30,1,96,111.380653


19. MENGEVALUASI MODEL

Interpretasi singkat:
MAE: rata-rata kesalahan prediksi dalam satuan quantity.
RMSE: memberi penalti lebih besar pada kesalahan yang besar.
R²: kemampuan model menjelaskan variasi qty.
Mendekati 1: performa lebih baik.
Mendekati 0: model belum menjelaskan pola dengan baik.
Nilai negatif: performa model sangat buruk dibandingkan prediksi rata-rata.

In [35]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("Hasil evaluasi model XGBoost")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


Hasil evaluasi model XGBoost
MAE  : 229.5656
RMSE : 617.4740
R²   : 0.3182


20. MENYIMPAN HASIL PREDIKSI

In [36]:
hasil_prediksi.to_csv(
    "hasil_prediksi_qty_xgboost.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Hasil prediksi tersimpan sebagai "
    "'hasil_prediksi_qty_xgboost.csv'"
)


Hasil prediksi tersimpan sebagai 'hasil_prediksi_qty_xgboost.csv'


In [38]:
import pandas as pd
import numpy as np

hasil_prediksi = pd.read_csv(
   "hasil_prediksi_qty_xgboost.csv",
   encoding="utf-8-sig"
)

# 1. Menentukan kolom produk

if "produk_bersih" in hasil_prediksi.columns:
    kolom_produk = "produk_bersih"
elif "produk_standar" in hasil_prediksi.columns:
    kolom_produk = "produk_standar"
elif "produk" in hasil_prediksi.columns:
    kolom_produk = "produk"
else:
    raise ValueError(
        "Kolom produk tidak ditemukan. "
        "Gunakan kolom produk_bersih, produk_standar, atau produk."
    )

# ============================================================
# 2. Membersihkan kolom yang digunakan
# ============================================================

data = hasil_prediksi.copy()

data["bulan"] = pd.to_numeric(
    data["bulan"],
    errors="coerce"
)

data["qty_aktual"] = pd.to_numeric(
    data["qty_aktual"],
    errors="coerce"
)

data["qty_prediksi"] = pd.to_numeric(
    data["qty_prediksi"],
    errors="coerce"
)

data[kolom_produk] = (
    data[kolom_produk]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Menghapus data yang tidak memiliki bulan atau nama produk
data = data.dropna(
    subset=["bulan", kolom_produk]
).copy()

# Menggunakan bulan 1 sampai 9 saja
data = data[
    data["bulan"].between(1, 9)
].copy()

# ============================================================
# 3. Agregasi quantity per bulan dan produk
# ============================================================

rekap_produk = (
    data
    .groupby(["bulan", kolom_produk], as_index=False)
    .agg(
        qty_aktual=("qty_aktual", "sum"),
        qty_prediksi=("qty_prediksi", "sum")
    )
)

# ============================================================
# 4. Membuat ranking berdasarkan quantity aktual
# ============================================================

ranking_aktual = (
    rekap_produk
    .sort_values(
        by=["bulan", "qty_aktual", kolom_produk],
        ascending=[True, False, True]
    )
    .copy()
)

ranking_aktual["peringkat"] = (
    ranking_aktual
    .groupby("bulan")
    .cumcount() + 1
)

# Format nama produk per bulan
ranking_aktual["produk_dan_qty"] = (
    ranking_aktual[kolom_produk]
    + " (qty: "
    + ranking_aktual["qty_aktual"].round(2).astype(str)
    + ")"
)

# ============================================================
# 5. Membuat ranking berdasarkan quantity prediksi
# ============================================================

ranking_prediksi = (
    rekap_produk
    .sort_values(
        by=["bulan", "qty_prediksi", kolom_produk],
        ascending=[True, False, True]
    )
    .copy()
)

ranking_prediksi["peringkat"] = (
    ranking_prediksi
    .groupby("bulan")
    .cumcount() + 1
)

# Format nama produk per bulan
ranking_prediksi["produk_dan_qty"] = (
    ranking_prediksi[kolom_produk]
    + " (qty: "
    + ranking_prediksi["qty_prediksi"].round(2).astype(str)
    + ")"
)

# ============================================================
# 6. Mengubah hasil menjadi format kolom bulan 1 sampai 9
# ============================================================

def buat_format_kolom(ranking):
    hasil = ranking.pivot(
        index="peringkat",
        columns="bulan",
        values="produk_dan_qty"
    )

    # Memastikan seluruh kolom bulan 1 sampai 9 tersedia
    hasil = hasil.reindex(
        columns=range(1, 10)
    )

    hasil.columns = [
        f"bulan_{int(kolom)}"
        for kolom in hasil.columns
    ]

    hasil = hasil.reset_index()

    return hasil


kesimpulan_aktual = buat_format_kolom(
    ranking_aktual
)

kesimpulan_prediksi = buat_format_kolom(
    ranking_prediksi
)

# ============================================================
# 7. Menampilkan hasil
# ============================================================

print("URUTAN PRODUK BERDASARKAN QTY AKTUAL")
display(kesimpulan_aktual)

print("URUTAN PRODUK BERDASARKAN QTY PREDIKSI")
display(kesimpulan_prediksi)

with pd.ExcelWriter(
    "kesimpulan_produk_per_bulan.xlsx",
    engine="openpyxl"
) as writer:

    kesimpulan_aktual.to_excel(
        writer,
        sheet_name="Ranking Aktual",
        index=False
    )

    kesimpulan_prediksi.to_excel(
        writer,
        sheet_name="Ranking Prediksi",
        index=False
    )

    rekap_produk.to_excel(
        writer,
        sheet_name="Rekap Quantity",
        index=False
    )

print(
    "File berhasil disimpan sebagai "
    "'kesimpulan_produk_per_bulan.xlsx'"
)



URUTAN PRODUK BERDASARKAN QTY AKTUAL


,peringkat,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9
0,1,NaN,NaN,NaN,NaN,NaN,G2G MOIST POME (qty: 480),SKIN1004 (qty: 7500),SCORA TONE UP (qty: 18624),BODYWASH NALPAMARA (qty: 3744)
1,2,NaN,NaN,NaN,NaN,NaN,AZARINE 50 ML (qty: 168),PURBASARI LULUR 5000 GR (qty: 5832),PURBASARI BODYSERUM (qty: 9600),SUNSCREEN WARDAH (qty: 3108)
2,3,NaN,NaN,NaN,NaN,NaN,FACE CREAM MOEL (qty: 115),G2G MICELLAR 300 ML PINK (qty: 4800),FACETOLOGY SUNSCREEN NORMAL (qty: 6360),FACETOLOGY TINTED SUNSCREEN (qty: 2680)
3,4,NaN,NaN,NaN,NaN,NaN,SUNSCREEN MOEL (qty: 115),PURBASARI BODYSERUM (qty: 4800),PURBASARI LULUR 1 KG (qty: 5160),PURBASARI BODYSERUM (qty: 2400)
4,5,NaN,NaN,NaN,NaN,NaN,MOIST PINK SKINTIFIC (qty: 96),MOIST BIRU SKINTIFIC (qty: 2784),MOIST BLUE G2G (qty: 4992),PURBASARI HABBATUS (qty: 2400)
5,6,NaN,NaN,NaN,NaN,NaN,<NA>,PURBASARI HABATTUS (qty: 2520),MOIST BIRU SKINTIFIC (qty: 3072),SUNSCREEN SCORA (qty: 2064)
6,7,NaN,NaN,NaN,NaN,NaN,<NA>,AZARINE ORANGE (qty: 1900),AZARINE KECIL (qty: 2790),SCORA FW (qty: 1980)
7,8,NaN,NaN,NaN,NaN,NaN,<NA>,AZARINE 50 ML (qty: 1518),SCORA FW (qty: 2340),SCORA TONE UP (qty: 1680)
8,9,NaN,NaN,NaN,NaN,NaN,<NA>,MOIST PINK SKINTIFIC (qty: 1344),BODYWASH NALPAMARA (qty: 2160),AMINO CLEANSER G2G (qty: 1632)
9,10,NaN,NaN,NaN,NaN,NaN,<NA>,BODYWASH NALPAMARA (qty: 1008),PURBASARI LULUR 500 GR (qty: 2040),G2G MOIST YUZA (qty: 1440)


URUTAN PRODUK BERDASARKAN QTY PREDIKSI


,peringkat,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9
0,1,NaN,NaN,NaN,NaN,NaN,G2G MOIST POME (qty: 218.84),SKIN1004 (qty: 11862.63),SCORA TONE UP (qty: 17616.41),BODYWASH NALPAMARA (qty: 3291.36)
1,2,NaN,NaN,NaN,NaN,NaN,SUNSCREEN MOEL (qty: 211.33),PURBASARI BODYSERUM (qty: 10414.22),PURBASARI BODYSERUM (qty: 11145.55),SCORA TONE UP (qty: 2829.06)
2,3,NaN,NaN,NaN,NaN,NaN,AZARINE 50 ML (qty: 180.79),MOIST BIRU SKINTIFIC (qty: 3421.39),FACETOLOGY SUNSCREEN NORMAL (qty: 6209.07),SUNSCREEN SCORA (qty: 2019.02)
3,4,NaN,NaN,NaN,NaN,NaN,MOIST PINK SKINTIFIC (qty: 111.38),HAIR LOTION MOEL (qty: 2663.25),MOIST BLUE G2G (qty: 5502.43),FACETOLOGY TINTED SUNSCREEN (qty: 1913.62)
4,5,NaN,NaN,NaN,NaN,NaN,FACE CREAM MOEL (qty: 105.67),MOIST PINK SKINTIFIC (qty: 2177.07),MOIST BIRU SKINTIFIC (qty: 4841.38),G2G MOIST KIWI (qty: 1545.46)
5,6,NaN,NaN,NaN,NaN,NaN,<NA>,BODYWASH NALPAMARA (qty: 1709.24),SCORA FW (qty: 4049.01),MOIST BIRU SKINTIFIC (qty: 1333.41)
6,7,NaN,NaN,NaN,NaN,NaN,<NA>,AZARINE ORANGE (qty: 1503.37),BODYWASH NALPAMARA (qty: 3623.15),SUNSCREEN WARDAH (qty: 1188.45)
7,8,NaN,NaN,NaN,NaN,NaN,<NA>,PURBASARI LULUR 5000 GR (qty: 1357.48),G2G MOIST KIWI (qty: 3212.98),G2G CENTELLA MOIST 30 GR (qty: 1156.06)
8,9,NaN,NaN,NaN,NaN,NaN,<NA>,G2G RETINOL MOIST (qty: 1315.77),G2G MOIST YUZA (qty: 2940.79),AMINO CLEANSER G2G (qty: 1104.31)
9,10,NaN,NaN,NaN,NaN,NaN,<NA>,PURBASARI HABATTUS (qty: 1301.13),AZARINE KECIL (qty: 2243.57),PURBASARI LULUR (qty: 1042.29)


File berhasil disimpan sebagai 'kesimpulan_produk_per_bulan.xlsx'


22. MENYIMPAN PIPELINE MODEL

mengunakan joblib untuk menyimpan pipeline lengkap, bukan hanya model XGBoost. Dengan demikian, OneHotEncoder dan preprocessing lainnya ikut tersimpan.

In [70]:
import joblib

# Metadata konfigurasi model
metadata_model = {
    "target": "qty",
    "kolom_numerik": [
        "tahun",
        "bulan",
        "hari",
        "hari_dalam_minggu",
        "harga_beli",
        "harga_jual"
    ],
    "kolom_kategorikal": [
        "pelanggan",
        "brand",
        "produk_bersih"
    ],
    "kolom_fitur": [
        "tahun",
        "bulan",
        "hari",
        "hari_dalam_minggu",
        "harga_beli",
        "harga_jual",
        "pelanggan",
        "brand",
        "produk_bersih"
    ]
}

# Menyimpan pipeline dan metadata
paket_model = {
    "pipeline": pipeline_xgb,
    "metadata": metadata_model
}

joblib.dump(
    paket_model,
    "model_qty_xgboost.joblib"
)

print("Model berhasil disimpan sebagai: model_qty_xgboost.joblib")


Model berhasil disimpan sebagai: model_qty_xgboost.joblib


23. Membaca model yang sudah disimpan

In [44]:
import joblib

paket_model = joblib.load(
    "model_qty_xgboost.joblib"
)

model_loaded = paket_model["pipeline"]
metadata = paket_model["metadata"]

print("Model berhasil dimuat.")
print("Fitur model:")
print(metadata["kolom_fitur"])


Model berhasil dimuat.
Fitur model:
['tahun', 'bulan', 'hari', 'hari_dalam_minggu', 'harga_beli', 'harga_jual', 'pelanggan', 'brand', 'produk_bersih']


24. Fungsi preprocessing data baru

Data baru minimal harus memiliki kolom:
text
tanggal
pelanggan
brand
produk_bersih
harga_beli
harga_jual
qty

25. Membaca data baru dan melakukan prediksi


In [45]:
data_baru = pd.read_csv(
    "hasil_pecah_tanggal.csv",
    encoding="utf-8-sig"
)
# Jika data baru berupa Excel:
# data_baru = pd.read_excel("data_transaksi_baru.xlsx")    
# 
pd.DataFrame(data_baru.head(10)) 

,bulan,tanggal,no_transaksi,pelanggan,qty,harga_beli,total_beli,harga_jual,total_jual,keuntungan,...,produk_bersih,produk_bersih.1,produk_standar,skor_kemiripan,skor_kedua,metode,status,tahun,hari,hari_dalam_minggu
0,1,2026-01-01,1,LUTFINUR,72,14000.0,1008000.0,14500,1044000,36000.0,...,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100.00,0.00,MATCH_EXACT,MATCH,2026,1,3
1,1,2026-01-02,1,HARIS,150,35000.0,5250000.0,36500,5475000,225000.0,...,SKINAQUA PUTIH,SKINAQUA PUTIH,NaN,69.09,69.09,MATCH_FUZZY,UNMATCHED,2026,2,4
2,1,2026-01-02,1,HARIS,144,16500.0,2376000.0,18000,2592000,216000.0,...,PERFECT WHITE AHA 50 ML,PERFECT WHITE AHA 50 ML,PERFECT WHITE AHA 50 ML,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
3,1,2026-01-02,2,KEDAIMUE,192,14000.0,2688000.0,14500,2784000,96000.0,...,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
4,1,2026-01-02,2,KEDAIMUE,96,29000.0,2784000.0,34500,3312000,528000.0,...,G2G MICELLAR 300 ML PINK,G2G MICELLAR 300 ML PINK,G2G MICELLAR 300 ML PINK,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
5,1,2026-01-02,3,SITA,360,14000.0,5040000.0,14500,5220000,180000.0,...,PURBASARI BODYSERUM,PURBASARI BODYSERUM,PURBASARI BODYSERUM,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
6,1,2026-01-02,4,WAFA,180,19000.0,3420000.0,20000,3600000,180000.0,...,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
7,1,2026-01-02,5,TERE,2376,15300.0,36352800.0,15700,37303200,950400.0,...,HANASUI LIPCREAM MATCHA,HANASUI LIPCREAM MATCHA,HANASUI LIPCREAM MATCHA,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
8,1,2026-01-02,6,SAZIRAH,192,24000.0,4608000.0,26000,4992000,384000.0,...,AMINO CLEANSER G2G,AMINO CLEANSER G2G,AMINO CLEANSER G2G,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4
9,1,2026-01-02,7,FENITA,60,19000.0,1140000.0,20000,1200000,60000.0,...,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,PURBASARI LULUR 500 GR,100.00,0.00,MATCH_EXACT,MATCH,2026,2,4


In [46]:
import pandas as pd
import numpy as np

data_baru = data_baru.copy()

# Memastikan komponen tanggal bertipe numerik
kolom_waktu = [
    "tahun",
    "bulan",
    "tanggal",
    "hari_dalam_minggu"
]

for kolom in kolom_waktu:
    data_baru[kolom] = pd.to_numeric(
        data_baru[kolom],
        errors="coerce"
    )

# Membuat tanggal lengkap dari tahun, bulan, dan tanggal
data_baru["tanggal_lengkap"] = pd.to_datetime(
    data_baru[
        ["tahun", "bulan", "tanggal"]
    ].rename(
        columns={
            "tahun": "year",
            "bulan": "month",
            "tanggal": "day"
        }
    ),
    errors="coerce"
)

# Kolom "hari" yang digunakan model adalah tanggal dalam bulan
data_baru["hari"] = data_baru["tanggal"]

kolom_numerik_model = [
    "tahun",
    "bulan",
    "hari",
    "hari_dalam_minggu",
    "harga_beli",
    "harga_jual"
]

for kolom in kolom_numerik_model:
    data_baru[kolom] = pd.to_numeric(
        data_baru[kolom],
        errors="coerce"
    )

kolom_kategorikal_model = [
    "pelanggan",
    "brand",
    "produk_bersih"
]

for kolom in kolom_kategorikal_model:
    data_baru[kolom] = (
        data_baru[kolom]
        .astype("object")
        .replace({pd.NA: np.nan})
        .fillna("TIDAK DIKETAHUI")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )



26. LAKUKAN PREDIKSI


In [47]:
X_baru = data_baru[
    metadata["kolom_fitur"]
].copy()
display(
    data_baru[
        [
            "tahun",
            "bulan",
            "tanggal",
            "tanggal_lengkap",
            "hari_dalam_minggu",
            "pelanggan",
            "brand",
            "produk_bersih",
        ]
    ].head()
)

data_baru.to_csv(
    "hasil_data_baru.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Hasil prediksi tersimpan sebagai "
    "'hasil_data_baru.csv'"
)




,tahun,bulan,tanggal,tanggal_lengkap,hari_dalam_minggu,pelanggan,brand,produk_bersih
0,2026,1,NaN,NaT,3,LUTFINUR,PURBASARI,PURBASARI BODYSERUM
1,2026,1,NaN,NaT,4,HARIS,SKINAQUA,SKINAQUA PUTIH
2,2026,1,NaN,NaT,4,HARIS,PERFECT,PERFECT WHITE AHA 50 ML
3,2026,1,NaN,NaT,4,KEDAIMUE,PURBASARI,PURBASARI BODYSERUM
4,2026,1,NaN,NaT,4,KEDAIMUE,G2G,G2G MICELLAR 300 ML PINK


Hasil prediksi tersimpan sebagai 'hasil_data_baru.csv'


In [48]:
import numpy as np
import pandas as pd

# Pastikan data baru memiliki fitur sesuai model
X_baru = data_baru[
    metadata["kolom_fitur"]
].copy()

# Melakukan prediksi
prediksi_qty = model_loaded.predict(X_baru)

# Mencegah hasil negatif dan membulatkan quantity
data_baru["qty_prediksi"] = (
    np.maximum(prediksi_qty, 0)
    .round()
    .astype(int)
)

# Menampilkan hasil
display(
    data_baru[
        [
            "tahun",
            "bulan",
            "tanggal",
            "no_transaksi",
            "pelanggan",
            "brand",
            "produk_bersih",
            "qty_prediksi"
        ]
    ].head()
)

data_baru.to_csv(
    "data_baru_dengan_qty_prediksi.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan.")


,tahun,bulan,tanggal,no_transaksi,pelanggan,brand,produk_bersih,qty_prediksi
0,2026,1,NaN,1,LUTFINUR,PURBASARI,PURBASARI BODYSERUM,923
1,2026,1,NaN,1,HARIS,SKINAQUA,SKINAQUA PUTIH,133
2,2026,1,NaN,1,HARIS,PERFECT,PERFECT WHITE AHA 50 ML,270
3,2026,1,NaN,2,KEDAIMUE,PURBASARI,PURBASARI BODYSERUM,348
4,2026,1,NaN,2,KEDAIMUE,G2G,G2G MICELLAR 300 ML PINK,118


File berhasil disimpan.


In [50]:
import pandas as pd

data = data_baru.copy()

# Pastikan tipe data sesuai
data["bulan"] = pd.to_numeric(
    data["bulan"],
    errors="coerce"
)

data["qty_prediksi"] = pd.to_numeric(
    data["qty_prediksi"],
    errors="coerce"
)

data["produk_bersih"] = (
    data["produk_bersih"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Hapus data yang tidak lengkap
data = data.dropna(
    subset=["bulan", "produk_bersih", "qty_prediksi"]
)

# Gunakan bulan 1 sampai 9
data = data[
    data["bulan"].between(1, 9)
]

# Total prediksi quantity setiap produk pada setiap bulan
rekap_prediksi = (
    data
    .groupby(
        ["bulan", "produk_bersih"],
        as_index=False
    )
    .agg(
        total_qty_prediksi=("qty_prediksi", "sum")
    )
)

# Urutkan dari paling laris
rekap_prediksi = rekap_prediksi.sort_values(
    by=[
        "bulan",
        "total_qty_prediksi",
        "produk_bersih"
    ],
    ascending=[
        True,
        False,
        True
    ]
)

# Membuat peringkat per bulan
rekap_prediksi["peringkat"] = (
    rekap_prediksi
    .groupby("bulan")
    .cumcount()
    .add(1)
)

# Format tampilan produk dan total quantity
rekap_prediksi["produk_dan_qty"] = (
    rekap_prediksi["produk_bersih"]
    + " ("
    + rekap_prediksi["total_qty_prediksi"]
        .round(0)
        .astype(int)
        .astype(str)
    + ")"
)

# Mengubah ranking menjadi kolom bulan
ranking_prediksi = (
    rekap_prediksi
    .pivot(
        index="peringkat",
        columns="bulan",
        values="produk_dan_qty"
    )
    .reindex(columns=range(1, 10))
)

ranking_prediksi.columns = [
    f"bulan_{bulan}"
    for bulan in ranking_prediksi.columns
]

ranking_prediksi = (
    ranking_prediksi
    .reset_index()
    .rename(columns={"peringkat": "peringkat"})
)

display(ranking_prediksi)


,peringkat,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9
0,1,PURBASARI BODYSERUM (40960),BODYWASH NALPAMARA (37947),BODYWASH NALPAMARA (19119),PURBASARI BODYSERUM (86316),PURBASARI LULUR 500 GR (11038),PURBASARI BODYSERUM (7582),SKIN1004 (11895),SCORA TONE UP (17941),BODYWASH NALPAMARA (3458)
1,2,PURBASARI LULUR 500 GR (32278),PURBASARI BODYSERUM (25665),PURBASARI BODYSERUM (16799),G2G SERUM RETINOL (19839),PURBASARI BODYSERUM (11025),PURBASARI LULUR 500 GR (7355),PURBASARI BODYSERUM (10438),PURBASARI BODYSERUM (11289),SCORA TONE UP (3201)
2,3,PURBASARI LULUR 1 KG (15132),PURBASARI LULUR 500 GR (24621),PURBASARI LULUR 5000 GR (6473),BODYWASH NALPAMARA (19369),BODYWASH NALPAMARA (6515),G2G MICELLAR 300 ML KUNING (6254),TIDAK DIKETAHUI (9720),FACETOLOGY SUNSCREEN NORMAL (6970),FACETOLOGY TINTED SUNSCREEN (2127)
3,4,BODYWASH NALPAMARA (12313),FACETOLOGY TINTED SUNSCREEN (15402),AZARINE ORANGE (3490),G2G MOIST POME (15412),SUNSCREEN SCORA (5367),BODYWASH NALPAMARA (4877),MOIST BIRU SKINTIFIC (3192),MOIST BIRU SKINTIFIC (5785),G2G MOIST KIWI (1670)
4,5,HANASUI BLUR TINT (11399),PURBASARI LULUR 1 KG (11740),PURBASARI LULUR 500 GR (2574),CUSHION G2G (12508),SUNSCREEN WARDAH (5105),MOIST BIRU SKINTIFIC (4206),HAIR LOTION MOEL (2699),MOIST BLUE G2G (5027),MOIST BIRU SKINTIFIC (1419)
...,...,...,...,...,...,...,...,...,...,...
79,80,<NA>,<NA>,<NA>,SKINTINT G2G (115),<NA>,<NA>,<NA>,<NA>,<NA>
80,81,<NA>,<NA>,<NA>,SETTING SPRAY G2G (114),<NA>,<NA>,<NA>,<NA>,<NA>
81,82,<NA>,<NA>,<NA>,DAY CREAM G2G (111),<NA>,<NA>,<NA>,<NA>,<NA>
82,83,<NA>,<NA>,<NA>,TWC G2G (110),<NA>,<NA>,<NA>,<NA>,<NA>


In [51]:
with pd.ExcelWriter(
    "ranking_produk_bulanan.xlsx",
    engine="openpyxl"
) as writer:

    ranking_prediksi.to_excel(
        writer,
        sheet_name="Ranking Prediksi",
        index=False
    )

    rekap_prediksi.to_excel(
        writer,
        sheet_name="Rekap Prediksi",
        index=False
    )



print("File berhasil disimpan sebagai ranking_produk_bulanan.xlsx")


File berhasil disimpan sebagai ranking_produk_bulanan.xlsx


In [52]:
paket_hasil = {
    # Model dan metadata sebelumnya
    "model": model_loaded,
    "metadata": metadata,

    # Data baru setelah ditambahkan prediksi
    "data_baru_dengan_prediksi": data_baru,

    # Rekap quantity per produk dan bulan
    "rekap_prediksi": rekap_prediksi,

    # Ranking produk per bulan
    "ranking_prediksi": ranking_prediksi
}

joblib.dump(
    paket_hasil,
    "model_dan_hasil_prediksi.joblib"
)

print(
    "Model, data prediksi, rekap, dan ranking berhasil disimpan."
)


Model, data prediksi, rekap, dan ranking berhasil disimpan.


In [53]:
hasil_loaded = joblib.load(
    "model_dan_hasil_prediksi.joblib"
)

model_loaded = hasil_loaded["model"]
metadata = hasil_loaded["metadata"]
data_baru = hasil_loaded["data_baru_dengan_prediksi"]
rekap_prediksi = hasil_loaded["rekap_prediksi"]
ranking_prediksi = hasil_loaded["ranking_prediksi"]

print("Seluruh paket hasil berhasil dimuat.")

display(data_baru.head())
display(rekap_prediksi.head())
display(ranking_prediksi.head())



Seluruh paket hasil berhasil dimuat.


,bulan,tanggal,no_transaksi,pelanggan,qty,harga_beli,total_beli,harga_jual,total_jual,keuntungan,...,produk_standar,skor_kemiripan,skor_kedua,metode,status,tahun,hari,hari_dalam_minggu,tanggal_lengkap,qty_prediksi
0,1,NaN,1,LUTFINUR,72,14000.0,1008000.0,14500,1044000,36000.0,...,PURBASARI BODYSERUM,100.00,0.00,MATCH_EXACT,MATCH,2026,NaN,3,NaT,923
1,1,NaN,1,HARIS,150,35000.0,5250000.0,36500,5475000,225000.0,...,NaN,69.09,69.09,MATCH_FUZZY,UNMATCHED,2026,NaN,4,NaT,133
2,1,NaN,1,HARIS,144,16500.0,2376000.0,18000,2592000,216000.0,...,PERFECT WHITE AHA 50 ML,100.00,0.00,MATCH_EXACT,MATCH,2026,NaN,4,NaT,270
3,1,NaN,2,KEDAIMUE,192,14000.0,2688000.0,14500,2784000,96000.0,...,PURBASARI BODYSERUM,100.00,0.00,MATCH_EXACT,MATCH,2026,NaN,4,NaT,348
4,1,NaN,2,KEDAIMUE,96,29000.0,2784000.0,34500,3312000,528000.0,...,G2G MICELLAR 300 ML PINK,100.00,0.00,MATCH_EXACT,MATCH,2026,NaN,4,NaT,118


,bulan,produk_bersih,total_qty_prediksi,peringkat,produk_dan_qty
57,1,PURBASARI BODYSERUM,40960,1,PURBASARI BODYSERUM (40960)
59,1,PURBASARI LULUR 500 GR,32278,2,PURBASARI LULUR 500 GR (32278)
58,1,PURBASARI LULUR 1 KG,15132,3,PURBASARI LULUR 1 KG (15132)
9,1,BODYWASH NALPAMARA,12313,4,BODYWASH NALPAMARA (12313)
41,1,HANASUI BLUR TINT,11399,5,HANASUI BLUR TINT (11399)


,peringkat,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9
0,1,PURBASARI BODYSERUM (40960),BODYWASH NALPAMARA (37947),BODYWASH NALPAMARA (19119),PURBASARI BODYSERUM (86316),PURBASARI LULUR 500 GR (11038),PURBASARI BODYSERUM (7582),SKIN1004 (11895),SCORA TONE UP (17941),BODYWASH NALPAMARA (3458)
1,2,PURBASARI LULUR 500 GR (32278),PURBASARI BODYSERUM (25665),PURBASARI BODYSERUM (16799),G2G SERUM RETINOL (19839),PURBASARI BODYSERUM (11025),PURBASARI LULUR 500 GR (7355),PURBASARI BODYSERUM (10438),PURBASARI BODYSERUM (11289),SCORA TONE UP (3201)
2,3,PURBASARI LULUR 1 KG (15132),PURBASARI LULUR 500 GR (24621),PURBASARI LULUR 5000 GR (6473),BODYWASH NALPAMARA (19369),BODYWASH NALPAMARA (6515),G2G MICELLAR 300 ML KUNING (6254),TIDAK DIKETAHUI (9720),FACETOLOGY SUNSCREEN NORMAL (6970),FACETOLOGY TINTED SUNSCREEN (2127)
3,4,BODYWASH NALPAMARA (12313),FACETOLOGY TINTED SUNSCREEN (15402),AZARINE ORANGE (3490),G2G MOIST POME (15412),SUNSCREEN SCORA (5367),BODYWASH NALPAMARA (4877),MOIST BIRU SKINTIFIC (3192),MOIST BIRU SKINTIFIC (5785),G2G MOIST KIWI (1670)
4,5,HANASUI BLUR TINT (11399),PURBASARI LULUR 1 KG (11740),PURBASARI LULUR 500 GR (2574),CUSHION G2G (12508),SUNSCREEN WARDAH (5105),MOIST BIRU SKINTIFIC (4206),HAIR LOTION MOEL (2699),MOIST BLUE G2G (5027),MOIST BIRU SKINTIFIC (1419)
